# PyTorch Lightning fine-tuning template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a surya model to implement DS application.

It focuses on the concept of defining a modified Surya model, loading its weigths, and using a PyTorch lightning training loop to train it

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**

It doesn't require having seen the baselines template, but they are meant to complement each other.  **_In fact they are on purpose almost identical!!!_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install lightning

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
torch.set_float32_matmul_precision('medium')



/home/ec2-user/surya_workshop/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [4]:
# The config is the single source of truth. load_flare_config() parses it into a typed
# object, exactly as the training script 3_finetune_template_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.template_sep.configs import load_flare_config

cfg = load_flare_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: solar_flare_forecasting


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [5]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


### Typed configuration, and how to extend it for your own task

`load_flare_config()` reads `configs/config_script.yaml` and returns a typed `TrainingConfig`.
This notebook and `3_finetune_template_1D.py` call the same function on the same file, so
there is no notebook-versus-script divergence to reason about.

| YAML section | Access in Python | Dataclass |
|---|---|---|
| `data:` | `cfg.data.*` | `FlareDataConfig` (this app) |
| `model:` | `cfg.model.*` | `ModelConfig` |
| `model.lora_config:` | `cfg.model.lora_config.*` | `LoraAdapterConfig` |
| `model.time_embedding:` | `cfg.model.time_embedding.*` | `TimeEmbeddingConfig` |
| `training:` | `cfg.learning_rate`, `cfg.batch_size`, … | `TrainingConfig` (flat) |
| `output:` | `cfg.output.*` | `OutputConfig` |
| `logging:` | `cfg.wandb_project`, `cfg.wandb_entity` | `TrainingConfig` (flat) |

**Everything except `FlareDataConfig` lives in `workshop_infrastructure/configs.py`** and is
shared by every downstream app. When you fork the template you do not copy that file. You
subclass `DataConfig` with your task's fields and bind `load_config` to it — the whole of
`downstream_apps/template/configs.py` is:

```python
@dataclass
class FlareDataConfig(DataConfig):
    flare_index_path: str = ""
    ds_time_column: str = "start_time"
    ds_time_tolerance: str = "4d"
    ds_match_direction: str = "forward"
    PATH_FIELDS = DataConfig.PATH_FIELDS + ("flare_index_path",)   # resolve it like a path

load_flare_config = partial(load_config, data_cls=FlareDataConfig)
```

Unknown keys are rejected rather than silently dropped: if you add a key to the YAML before
adding the field, you get an error naming the key and listing the valid ones.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additonal parameters relevant to the downstream application.  Here we focus in particular to the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is normalization.  Here we use a log normalization on xray flux that will act as the output target.  Making log10(xray_flux) strictly positive and having 66% of its values between 0 and 1

In this case we will define both a training and a validation dataset using the indices pointed at in the config

**_Important:  In this notebook we sets max_number_of_samples=6 to potentially avoid going through the whole dataset as we explore it.  Keep in mind this for the future in case the database seems smaller than you expect_**


In [ ]:
!pip install scikit-image

In [ ]:
!pip install hdf5plugin

In [6]:
from downstream_apps.template_sep.datasets.sep_dataset import SEPDSDataset

In [7]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the flare-specific arguments are passed here, which is exactly
# the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders

cfg.batch_size=1

train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    SEPDSDataset,
    scalers=scalers,
    num_workers=0,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    max_number_of_samples=3,
    ds_sep_index_path=cfg.data.train_data_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


train: 3 samples | val: 3 samples | batch_size: 1


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


## Initialize the HelioSpectformer model

This is the main difference beteween the notebook that trains the simple model and the one that fine-tunes Surya.  

In the case of the finetuning exercise one of the main differences between DS applications is the dimensionality of the output.  In this notebook we use a modified HelioSpectformer that projects into a 1D space. 

**_IMPORTANT: If your DS application is 2D you need to use the HelioSpectformer2D_**

In [8]:
from workshop_infrastructure.models.finetune_models import HelioSpectformer1D

/home/ec2-user/surya_workshop/.venv/lib64/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Now the config file really comes into bear. The Spectformer has a metric ton of hyperparameters

In [9]:
# HelioSpectformer1D has a long list of architecture arguments, and all of them come
# straight from the model: section of the config. from_config() does that mapping, so the
# backbone can never drift out of sync with the checkpoint it is about to load.
#
# Arguments that are not part of ModelConfig (dtype, and anything from the training:
# section) are passed as explicit overrides.

cfg.model.checkpoint_layers = None

model = HelioSpectformer1D.from_config(
    cfg.model,
    num_outputs=1,
    pooling="class_token",
    dtype=cfg.dtype,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
)

In [ ]:
!pip install h5netcdf

In [10]:
batch = next(iter(train_data_loader))
print(type(batch))
print(batch.keys())          # if it's a dict
# or print([x.shape for x in batch])  # if it's a tuple/list

['Unnamed: 0', 'window_begin', 'window_end', '>10.0 MeV 10.0 pfu SEP Start Time', 'SEP', 'flare_strength', 'Flare_label', 'timestep', 'flare_peak_time', 'flare_end_time', 'flare_fl_cls', 'flare_Strength', 'path', 'match_type', 'present', 'ds_index', 'sep_label']
<class 'dict'>
dict_keys(['ts', 'time_delta_input', 'sep', 'ds_index', 'flare_strength'])


In [10]:
"""
import torch
import torch.nn as nn
from tqdm import tqdm  # pip install tqdm if not available

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

num_epochs = 1

for epoch in range(num_epochs):

    # ── Training ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_data_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for batch in pbar:
        optimizer.zero_grad()

        labels = batch['sep'].float()
        logits = model(batch)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        # Accumulate metrics
        train_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).long()
        train_correct += (preds == labels.long()).sum().item()
        train_total += labels.size(0)

        # Live update on the progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc":  f"{train_correct / train_total:.3f}",
        })

    avg_train_loss = train_loss / len(train_data_loader)
    avg_train_acc  = train_correct / train_total

    # ── Validation ────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        pbar_val = tqdm(val_data_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]  ")
        for batch in pbar_val:
            labels = batch['sep'].float()
            logits = model(batch)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).long()
            val_correct += (preds == labels.long()).sum().item()
            val_total += labels.size(0)

            pbar_val.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc":  f"{val_correct / val_total:.3f}",
            })

    avg_val_loss = val_loss / len(val_data_loader)
    avg_val_acc  = val_correct / val_total

    # ── Epoch summary ─────────────────────────────────────────
    print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
    print(f"  Train — Loss: {avg_train_loss:.4f}  Acc: {avg_train_acc:.3f}")
    print(f"  Val   — Loss: {avg_val_loss:.4f}  Acc: {avg_val_acc:.3f}\n")
"""

Epoch 1/1 [Train]:   0%|          | 0/3 [00:00<?, ?it/s]

['Unnamed: 0', 'window_begin', 'window_end', '>10.0 MeV 10.0 pfu SEP Start Time', 'SEP', 'flare_strength', 'Flare_label', 'timestep', 'flare_peak_time', 'flare_end_time', 'flare_fl_cls', 'flare_Strength', 'path', 'match_type', 'present', 'ds_index', 'sep_label']


: 

## Load model weights

Here we load the pre-trained checkpoint and load the weights.  The exercise of loading follows the idea of us as many of the weights as possible.  This is accomplished through the filtered_checkpoint_state.   It checks to see if the pretrained model's layers match those of your finetuning architecture.   It also checks that all your dimensions across layers check out.   If something does not work those paramameters are left in their random initialization. 

In [11]:
# The checkpoint was saved from HelioSpectFormer directly, so its keys are flat
# (e.g. "embedding.proj.weight"), while the fine-tuning model nests the backbone under
# "backbone.*". load_pretrained_weights() tries both spellings and reports how many
# tensors matched — a low count means the architecture does not match the checkpoint.
from workshop_infrastructure.utils import load_pretrained_weights

load_pretrained_weights(model, cfg.model.pretrained_path)


Loading pretrained weights from /home/ec2-user/surya_workshop/downstream_apps/template_sep/assets/surya.366m.v1.pt.
Loaded 156 / 159 pretrained weights.


## To LoRA or not to Lora

This cell gives you two options.  On the one hand we have the classic freezing of the backbone (the initial layers of the model).   On the other hand we have the use of a LoRA.

LoRas have been a remarkable addition to our arsenal of models.   They have the advantage of keeping pretty much the entire model intact and only add broad modifications to weights as needed.

**What actually trains.** In the LoRA regime it is the adapters *and* the whole fine-tuning head. That second part is easy to get wrong: PEFT freezes every parameter it does not recognise as an adapter, so unless the head is explicitly handed to it as `modules_to_save`, the adapters end up fitting a **frozen, randomly initialised readout** — and the loss still goes down, so the training curve looks perfectly healthy. `apply_peft_lora()` avoids this by discovering every `head_*` attribute on the model and marking it trainable. If you add your own head layer, give it a `head_` prefix or it will be silently frozen (the helper checks this at startup and tells you what to rename).

**Where the adapters go.** `fc1`/`fc2` in all ten blocks, plus `attn.qkv` and `attn.proj` in the eight attention blocks. The spectral `complex_weight`, `attn.to_dynamic_projection`, and the patch embedding are never adapted.

Surya fuses query, key and value into a single `nn.Linear(1280, 3840)`, so one adapter covers all three at once: they share the `8×1280` matrix `A` and each gets its own `1280×8` slice of `B`. Their combined rank is at most 8 — which is *not* the same as giving q, k and v three independent rank-8 adapters.

Run the cell below and check the printout: the LoRA regime should report **3,157,761** trainable parameters (1,515,520 of adapters + 1,642,241 of head), and the linear probe **1,642,241**.

In [12]:
# Three fine-tuning regimes, all selected from the model: section of the config:
#
#   use_lora: true                          -> LoRA adapters + the whole head (default)
#   use_lora: false, freeze_backbone: true  -> linear probe: only the head trains
#   use_lora: false, freeze_backbone: false -> full fine-tuning of all 366M parameters
#
# freeze_backbone is ignored when use_lora is true: PEFT freezes everything, then
# re-enables the adapters and every head_* module.
#
# 3_finetune_template_1D.py applies exactly this logic in build_model().
if cfg.model.freeze_backbone:
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = False

if cfg.model.use_lora:
    # Prints the adapted modules and the trainable head modules it discovered.
    model = apply_peft_lora(model, cfg.model.lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Applying PEFT LoRA: r=8, alpha=8, dropout=0.1, modules=['attn.qkv', 'attn.proj', 'fc1', 'fc2']
[LoRA] Adapted modules (36):
[LoRA]   backbone.backbone.blocks_attention.0.attn.proj
[LoRA]   backbone.backbone.blocks_attention.0.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.1.attn.proj
[LoRA]   backbone.backbone.blocks_attention.1.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.2.attn.proj
[LoRA]   backbone.backbone.blocks_attention.2.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.3.attn.proj
[LoRA]   backbone.backbone.blocks_attention.3.attn.qkv
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc2


We can now test that this model manipulates a batch as expected and returns an estimate of flare intensity as we did for the simple baseline.

We pass the input stack 'ts' to the model to transform it into our regression output.   Note that since this model was trained for a different task, it's likely it won't perform very well.  As with the simple baseline, this only acts as a test that our model forward doesn't have dimension problems.

Dimension problemns are the dominant source of error in this kind of work.

Note that our output has now the size of our batch.

In [13]:
model = model.to(torch.bfloat16)
batch['ts'] = batch['ts'].to(torch.bfloat16)

with torch.no_grad():
    output = model(batch)

In [ ]:
batch = next(iter(train_data_loader))
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f"{k}: {v.shape}, dtype={v.dtype}")
    else:
        print(f"{k}: {type(v)} = {v}")

## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantitification of error, which in turn shifts the weights towards better pefrorming models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics do you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can be later use in a training script

In [14]:
from downstream_apps.template_sep.metrics.template_metrics_sep import SEPMetrics

In [15]:
train_loss_metrics = SEPMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same MSE as train_loss — override FlareMetrics.val_loss to change it.
val_loss_metrics = SEPMetrics("val_loss")
train_evaluation_metrics = SEPMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = SEPMetrics("val_metrics")

Now they can be evaluated in our model's output and our ground truth.   First the loss that actually will backpropagate, in this case Mean Squared Errror

In [19]:
output = torch.sigmoid(output).float()   # logits → probs in [0,1]
print("probs:", output)                  # should now be between 0 and 1

train_loss_metrics(output, batch["sep"])

probs: tensor([0.4443])


({'bce': tensor(0.8112)}, [1])

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it her as an example, but oftentimes is better to put the diagnostics only in the validation evaluation metrics.

Here we are caclulating the Root Relative Squared Error https://lightning.ai/docs/torchmetrics/stable/regression/rse.html 

A value below one means the prediction is better than predicting the average.  It is unlikely that this metric will be lower than one with a randomly initialized model

In [20]:
train_evaluation_metrics(output, batch["sep"])

({'accuracy': tensor(0.)}, [1])

In the validation evaluation metrics we report both MSE and RRSE

In [21]:
validation_evaluation_metrics(output, batch["sep"])

({'accuracy': tensor(0.)}, [1])

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the FlareLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

**_Note that it is the same Lightning module we used for the baseline!!_**

In [22]:
from downstream_apps.template_sep.lightning_modules.sep_simpl_baseline import SEPLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [ ]:
L.seed_everything(42, workers=True)

## Intialize Lightning module

Now we properly initalize the Lightning module to enable training, including passing the dictionary of metrics

In [23]:
metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

lit_model = SEPLightningModule(model, metrics, lr=cfg.learning_rate, batch_size=batch_size)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [25]:
!pip install -U 'wandb>=0.12.10'

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 178.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 218.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [28]:
project_name = cfg.wandb_project
run_name = "finetune_experiment_1"  # give your run a descriptive name

"""
wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)
"""

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).


**Note that in this notebook we also set a mixed precision to reduce the model's footprint in memory.**

In [29]:
max_epochs = 2

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    precision="bf16-mixed", 
    logger=[csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Fit the model

Finally we fit the model.  We pass the Lighting module, and our dataloaders.

In [30]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/ec2-user/surya_workshop/.venv/lib64/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type      | Params | Mode  | FLOPs
----------------------------------------------------
0 | model | PeftModel | 362 M  | train | 0    
----------------------------------------------------
3.2 M     Trainable params
359 M     Non-trainable params
362 M     Total params
1,449.868 Total estimated model params size (MB)
540       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/ec2-user/surya_workshop/.venv/lib64/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/ec2-user/surya_workshop/.venv/lib64/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


['Unnamed: 0', 'window_begin', 'window_end', '>10.0 MeV 10.0 pfu SEP Start Time', 'SEP', 'flare_strength', 'Flare_label', 'timestep', 'flare_peak_time', 'flare_end_time', 'flare_fl_cls', 'flare_Strength', 'path', 'match_type', 'present', 'ds_index', 'sep_label']
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: torch.nn.functional.binary_cross_entropy and torch.nn.BCELoss are unsafe to autocast.
Many models use a sigmoid layer right before the binary cross entropy layer.
In this case, combine the two layers using torch.nn.functional.binary_cross_entropy_with_logits
or torch.nn.BCEWithLogitsLoss.  binary_cross_entropy_with_logits and BCEWithLogits are
safe to autocast.

## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and DS into an end-2-end training loop and we are ready to experiment!